# 99 Final: the chosen model against matrix pricing

The submission notebook. It runs top to bottom from the panel and produces the headline table and the quality
gate verdict.

| Mode | `OPEN_TEST` | Fits on | Scores on | When |
|---|---|---|---|---|
| Dry run | `False` | train | valid | Before test is opened: checks the whole pipeline and reproduces the valid score |
| Final | `True` | train + valid | test (once) | After the choice is logged in `RUN_LOG.md` (P13) |

The gate (`config.yaml`, `quality_gate`): at least 10% lower MAE than matrix pricing re-marked on each snapshot
date, with a 95% issuer-bootstrap interval above zero, in IG and in HY.

`OPEN_TEST = True` writes `notebooks/test_opened.json` with the model's fingerprint (every setting, the feature
list, the fitting months and the data checksum). Commit and push it straight away so the whole group shares it.
Re-running the same model is fine; running a different model against test stops with an error.

In [1]:
import pandas as pd
from sklearn.pipeline import Pipeline
import xgboost as xgb

from ml2.baseline import fit_matrix_frozen, matrix_same_date, median_baseline, predict_matrix_frozen
from ml2.config import load_config, resolve
from ml2.data import check_panel, load_panel
from ml2.evaluate import gate_report, metrics_by_stratum
from ml2.features import make_preprocessor, xy
from ml2.final import model_spec, on_log_spread, open_test_once

cfg = load_config()
TARGET = cfg["problem"]["target"]

OPEN_TEST = False  # True only once the choice is logged in RUN_LOG.md (P13)

## 1. The chosen model

Chosen on validation and logged in `RUN_LOG.md` before test was opened (P13): XGBoost with the settings from
notebook 03's grid search (500 trees, depth 10, learning rate 0.03, 80% of rows and 70% of columns per tree),
fitted on log(spread + 101) instead of the spread.

Why this one. On valid, against the bar of 21.2 / 88.4 bp (IG / HY):

| Model | IG | HY |
|---|---|---|
| XGBoost on log(spread + 101) | 13.9 | 46.0 |
| MLP (notebook 05) | 13.7 | 56.8 |
| XGBoost on the spread (notebook 03) | 15.2 | 53.8 |
| Random forest (notebook 03) | 15.4 | 62.1 |

It ties the MLP in IG (0.2 bp apart) and is 11 bp better in HY. The log target helps because spreads are
right-skewed, most of all in HY: the model stops spending its effort on a few very wide bonds.

In [2]:
FINAL_NAME = "XGBoost on log(spread + 101)"


def make_final_model():
    return on_log_spread(Pipeline([
        ("pre", make_preprocessor(scale=False, one_hot=False)),
        ("est", xgb.XGBRegressor(objective="reg:squarederror", n_estimators=500, max_depth=10, learning_rate=0.03,
                                 subsample=0.8, colsample_bytree=0.7, random_state=42, n_jobs=-1)),
    ]))


SPEC = model_spec(FINAL_NAME, make_final_model(), ["train", "valid"])
print(SPEC[:300], "...")

{"estimator": "TransformedTargetRegressor", "features": ["years_to_maturity", "maturity_bucket", "coupon_pct", "grade_bucket", "is_144a", "debt_type", "is_subordinated", "n_trades", "log_par_volume", "has_inst_trade", "trade_days_in_month", "log_par_volume_month", "n_bonds_issuer", "issuer_log_par_v ...


## 2. Data

In [3]:
check_panel()
if OPEN_TEST:
    print(open_test_once(SPEC))  # before test is loaded: records or checks the model
fit_parts, score_part = (["train", "valid"], "test") if OPEN_TEST else (["train"], "valid")

panel = load_panel(partitions=fit_parts + [score_part], include_test=OPEN_TEST)
fit = panel[panel["partition"].isin(fit_parts)]
score = panel[(panel["partition"] == score_part) & panel[TARGET].notna()].copy()
print(f"Fit on {fit_parts}: {len(fit):,} rows. Score on {score_part}: {len(score):,} rows")

Fit on ['train']: 119,456 rows. Score on valid: 28,569 rows


## 3. Fit and predict

In [4]:
X_fit, y_fit = xy(fit)
X_score, _ = xy(score)

model = make_final_model().fit(X_fit, y_fit)
score["pred_final"] = model.predict(X_score)

## 4. Baselines

Matrix pricing re-marked on each date is the gate baseline: each bond priced from the same-day spreads of other
companies' bonds in its grade x maturity x sector cell. The median and frozen matrix are reference rows.

In [5]:
score["pred_matrix_same_date"] = matrix_same_date(score)
score["pred_matrix_frozen"] = predict_matrix_frozen(fit_matrix_frozen(fit), score)
score["pred_median"] = median_baseline(fit, score)

## 5. Headline table: MAE in bp by grade

In [6]:
ROWS = {
    "pred_final": FINAL_NAME,
    "pred_matrix_same_date": "Matrix pricing, re-marked each date (gate baseline)",
    "pred_matrix_frozen": "Matrix pricing, frozen at the fitting months",
    "pred_median": "Median spread",
}
headline = pd.concat(
    [metrics_by_stratum(score, TARGET, col, "grade_bucket").assign(model=name) for col, name in ROWS.items()]
)
table = headline.pivot(index="model", columns="stratum", values="mae_bps").loc[list(ROWS.values())]
table = table[["IG", "HY", "ALL"]].round(1)
table

stratum,IG,HY,ALL
model,,,
XGBoost on log(spread + 101),13.9,46.0,16.3
"Matrix pricing, re-marked each date (gate baseline)",21.2,88.4,26.2
"Matrix pricing, frozen at the fitting months",23.2,86.1,27.9
Median spread,29.8,126.9,37.1


## 6. Quality gate

In [7]:
ok, gate = gate_report(score, TARGET, "pred_final", "pred_matrix_same_date", cfg["quality_gate"])
print(f"{score_part}: gate {'PASSED' if ok else 'NOT PASSED'}")
gate.round(3)

valid: gate PASSED


,model,baseline,improvement,pass,ci_low,ci_high
stratum,,,,,,
IG,13.893,21.177,0.344,True,0.298,0.393
HY,46.005,88.362,0.479,True,0.389,0.554


In [8]:
mode = "final_test" if OPEN_TEST else "dryrun_valid"
out = resolve("reports")
table.to_csv(out / f"headline_{mode}.csv")
gate.to_csv(out / f"gate_{mode}.csv")
print(f"Saved reports/headline_{mode}.csv and reports/gate_{mode}.csv. Add a line to notebooks/RUN_LOG.md.")

Saved reports/headline_dryrun_valid.csv and reports/gate_dryrun_valid.csv. Add a line to notebooks/RUN_LOG.md.
